# 03 - Data Cleaning

**Bronze tables → Cleaning → Silver tables**

This notebook performs the actual cleaning identified in `02_data_validation`.
Bronze tables are never modified.

## 1. Load Bronze Tables

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

stores_df = spark.table("stores_bronze")
products_df = spark.table("products_bronze")
suppliers_df = spark.table("suppliers_bronze")
sales_df = spark.table("sales_bronze")
inventory_df = spark.table("inventory_bronze")
purchase_orders_df = spark.table("purchase_orders_bronze")

## 2. Clean Stores

Missing manager names are replaced with `Unknown`.

In [0]:
stores_clean = stores_df.withColumn(
    "manager_name",
    F.when(
        F.col("manager_name").isNull() | (F.trim(F.col("manager_name")) == ""),
        "Unknown"
    ).otherwise(F.trim(F.col("manager_name")))
)

display(stores_clean)

## 3. Clean Suppliers

Standardize phone numbers to a 10-digit format by removing separators and the Indian country code when present.

Missing contact persons are replaced with `Unknown`.

In [0]:
suppliers_clean = (
    suppliers_df
    .withColumn(
        "contact_person",
        F.when(
            F.col("contact_person").isNull() | (F.trim(F.col("contact_person")) == ""),
            "Unknown"
        ).otherwise(F.trim(F.col("contact_person")))
    )
    .withColumn("phone", F.regexp_replace(F.col("phone"), r"[^0-9]", ""))
    .withColumn(
        "phone",
        F.when(
            F.col("phone").rlike(r"^91[0-9]{10}$"),
            F.substring(F.col("phone"), 3, 10)
        ).otherwise(F.col("phone"))
    )
)

display(suppliers_clean)

## 4. Clean Products

Cleaning decisions:
- Missing category → `Unknown`.
- Missing supplier ID cannot be safely inferred, so those products are removed from Silver.
- Missing or non-positive unit prices are invalid, so those products are removed from Silver.

In [0]:
products_df.filter(
        (F.col("unit_price").isNull()) |
        (F.col("unit_price") <= 0)
    ).show()

In [0]:
products_clean = (
    products_df
    .withColumn(
        "category",
        F.when(
            F.col("category").isNull() | (F.trim(F.col("category")) == ""),
            "Unknown"
        ).otherwise(F.trim(F.col("category")))
    )
    .filter(F.col("supplier_id").isNotNull())
    .filter(F.col("unit_price").isNotNull() & (F.col("unit_price") > 0))
)

display(products_clean)

## 5. Clean Sales

Cleaning decisions:
- Remove duplicate business records, keeping the lowest `transaction_id`.
- Remove records with invalid store/product references.
- Remove records with non-positive quantity or price.
- Recalculate `total_amount` from `quantity_sold * unit_price`.
- Keep only sales dated within the project's 2024 data range.

In [0]:
sales_window = Window.partitionBy(
    "store_id",
    "product_id",
    "quantity_sold",
    "unit_price",
    "total_amount",
    "sale_date"
).orderBy("transaction_id")

sales_clean = (
    sales_df
    .withColumn("_row_number", F.row_number().over(sales_window))
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
    .filter(F.col("quantity_sold").isNotNull() & (F.col("quantity_sold") > 0))
    .filter(F.col("unit_price").isNotNull() & (F.col("unit_price") > 0))
    .filter(
        (F.col("sale_date") >= F.lit("2024-01-01")) &
        (F.col("sale_date") <= F.lit("2024-12-31"))
    )
    .withColumn(
        "total_amount",
        F.round(F.col("quantity_sold") * F.col("unit_price"), 2)
    )
)

valid_store_ids = stores_clean.select("store_id").distinct()
valid_product_ids = products_clean.select("product_id").distinct()

sales_clean = (
    sales_clean
    .join(valid_store_ids, "store_id", "inner")
    .join(valid_product_ids, "product_id", "inner")
)

display(sales_clean)

## 6. Clean Inventory

Zero stock is valid and is retained. Negative or missing quantities are invalid and removed.

Only valid store-product combinations are retained.

In [0]:
inventory_clean = (
    inventory_df
    .filter(
        F.col("quantity_on_hand").isNotNull() &
        (F.col("quantity_on_hand") >= 0)
    )
    .join(valid_store_ids, "store_id", "inner")
    .join(valid_product_ids, "product_id", "inner")
)

display(inventory_clean)

## 7. Clean Purchase Orders

Cleaning decisions:
- Missing delivery dates are retained because they represent pending orders.
- Delivery dates earlier than order dates are invalid and removed.
- Order and delivery dates are restricted to the project's 2024 data range.
- Only valid supplier IDs are retained.

In [0]:
valid_supplier_ids = suppliers_clean.select("supplier_id").distinct()

purchase_orders_clean = (
    purchase_orders_df
    .filter(
        (F.col("order_date") >= F.lit("2024-01-01")) &
        (F.col("order_date") <= F.lit("2024-12-31"))
    )
    .filter(
        F.col("delivery_date").isNull() |
        (
            (F.col("delivery_date") >= F.lit("2024-01-01")) &
            (F.col("delivery_date") <= F.lit("2024-12-31")) &
            (F.col("delivery_date") >= F.col("order_date"))
        )
    )
    .join(valid_supplier_ids, "supplier_id", "inner")
)

display(purchase_orders_clean)

## 8. Final Silver Checks

In [0]:
print("Bronze row counts")
print("stores:", stores_df.count())
print("products:", products_df.count())
print("suppliers:", suppliers_df.count())
print("sales:", sales_df.count())
print("inventory:", inventory_df.count())
print("purchase_orders:", purchase_orders_df.count())
print()

print("Silver row counts")
print("stores:", stores_clean.count())
print("products:", products_clean.count())
print("suppliers:", suppliers_clean.count())
print("sales:", sales_clean.count())
print("inventory:", inventory_clean.count())
print("purchase_orders:", purchase_orders_clean.count())

print("\nRemaining invalid product prices:",
      products_clean.filter(F.col("unit_price") <= 0).count())

print("Remaining duplicate sales business keys:",
      sales_clean.groupBy(
          "store_id", "product_id", "quantity_sold",
          "unit_price", "total_amount", "sale_date"
      ).count().filter(F.col("count") > 1).count())

print("Remaining invalid inventory quantities:",
      inventory_clean.filter(F.col("quantity_on_hand") < 0).count())

print("Remaining invalid PO dates:",
      purchase_orders_clean.filter(
          F.col("delivery_date").isNotNull() &
          (F.col("delivery_date") < F.col("order_date"))
      ).count())

## 9. Write Silver Tables


In [0]:
stores_clean.write.mode("overwrite").format("delta").saveAsTable("stores_silver")
products_clean.write.mode("overwrite").format("delta").saveAsTable("products_silver")
suppliers_clean.write.mode("overwrite").format("delta").saveAsTable("suppliers_silver")
sales_clean.write.mode("overwrite").format("delta").saveAsTable("sales_silver")
inventory_clean.write.mode("overwrite").format("delta").saveAsTable("inventory_silver")
purchase_orders_clean.write.mode("overwrite").format("delta").saveAsTable("purchase_orders_silver")

## 10. Cleaning Summary

| Table | Cleaning decision |
|---|---|
| Stores | Missing manager → `Unknown` |
| Suppliers | Missing contact → `Unknown`; phone format standardized |
| Products | Missing category → `Unknown`; missing/non-positive price removed; missing supplier removed |
| Sales | Duplicate business records removed; invalid quantity/price/date/reference removed; total amount recalculated |
| Inventory | Negative/missing quantity removed; zero stock retained; invalid references removed |
| Purchase Orders | Pending NULL delivery dates retained; invalid dates/references removed |
| Bronze | Not modified |
| Silver | Cleaned tables written as Delta tables |